# Konversi dan Ekstraksi Teks


In [2]:
import os
from pypdf import PdfReader

# Tentukan path folder data raw
raw_data_dir = "../data/raw/"

# Ambil semua file yang berakhiran .pdf di dalam folder tersebut
pdf_files = [f for f in os.listdir(raw_data_dir) if f.endswith('.pdf')]

print(f"Menemukan {len(pdf_files)} file PDF untuk dikonversi.")

# Proses konversi setiap file PDF
for index, pdf_file in enumerate(pdf_files, start=1):
    pdf_path = os.path.join(raw_data_dir, pdf_file)
    
    # Tentukan nama file .txt keluaran (misal: case_001.txt)
    txt_filename = f"case_{index:03d}.txt"
    txt_path = os.path.join(raw_data_dir, txt_filename)
    
    try:
        # Membaca file PDF
        reader = PdfReader(pdf_path)
        extracted_text = []
        
        # Ekstrak teks dari setiap halaman PDF
        for page in reader.pages:
            text = page.extract_text()
            if text:
                extracted_text.append(text)
        
        # Gabungkan teks seluruh halaman menjadi satu string
        full_text = "\n".join(extracted_text)
        
        # Simpan hasil ekstraksi teks ke file .txt
        with open(txt_path, "w", encoding="utf-8") as txt_file:
            txt_file.write(full_text)
            
        print(f"✅ Berhasil mengonversi: {pdf_file} -> {txt_filename}")
        
    except Exception as e:
        print(f"❌ Gagal memproses {pdf_file}. Error: {e}")

print("\nProses konversi selesai!")

Menemukan 36 file PDF untuk dikonversi.
✅ Berhasil mengonversi: case_001.pdf -> case_001.txt
✅ Berhasil mengonversi: case_002.pdf -> case_002.txt
✅ Berhasil mengonversi: case_003.pdf -> case_003.txt
✅ Berhasil mengonversi: case_004.pdf -> case_004.txt
✅ Berhasil mengonversi: case_005.pdf -> case_005.txt
✅ Berhasil mengonversi: case_006.pdf -> case_006.txt
✅ Berhasil mengonversi: case_007.pdf -> case_007.txt
✅ Berhasil mengonversi: case_008.pdf -> case_008.txt
✅ Berhasil mengonversi: case_009.pdf -> case_009.txt
✅ Berhasil mengonversi: case_010.pdf -> case_010.txt
✅ Berhasil mengonversi: case_011.pdf -> case_011.txt
✅ Berhasil mengonversi: case_012.pdf -> case_012.txt
✅ Berhasil mengonversi: case_013.pdf -> case_013.txt
✅ Berhasil mengonversi: case_014.pdf -> case_014.txt
✅ Berhasil mengonversi: case_015.pdf -> case_015.txt
✅ Berhasil mengonversi: case_016.pdf -> case_016.txt
✅ Berhasil mengonversi: case_017.pdf -> case_017.txt
✅ Berhasil mengonversi: case_018.pdf -> case_018.txt
✅ Berh

# Pembersihan dan Validasi

In [5]:
import os
import re

# Tentukan path folder data raw
raw_data_dir = "../data/raw/"

# Ambil semua file .txt hasil konversi awal
txt_files = [f for f in os.listdir(raw_data_dir) if f.endswith('.txt') and f.startswith('case_')]
txt_files.sort()

print(f"Menemukan {len(txt_files)} file teks untuk dibersihkan.\n")

# Log untuk memantau kualitas data
log_lines = ["=== LOG FILE PEMBERSIHAN DATA PUTUSAN ==="]

for txt_file in txt_files:
    file_path = os.path.join(raw_data_dir, txt_file)
    
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()
    
    len_original = len(text)
    
    # --- PROSES PEMBERSIHAN YANG AMAN ---
    
    # 1. Ubah ke lower-case agar seragam
    cleaned_text = text.lower()
    
    # 2. Hapus Header, Footer, Halaman, dan Watermark Mahkamah Agung
    # Menghapus baris berulang yang tidak penting tanpa merusak kalimat utama
    pola_sampah = [
        r"mahkamah\s+agung\s+republik\s+indonesia",
        r"direktorat\s+jenderal\s+badan\s+peradilan.*",
        r"kepaniteraan\s+mahkamah\s+agung.*",
        r"halaman\s+\d+\s+dari\s+\d+\s+putusan.*",
        r"hal\s+\d+\s+put\s+no.*",
        r"disclaimer.*",
        r"dalam\s+hal\s+anda\s+menemukan\s+incompabilitas.*"
    ]
    for pola in pola_sampah:
        cleaned_text = re.sub(pola, "", cleaned_text)
    
    # 3. Normalisasi Spasi Ganda (Merapikan teks agar tidak renggang)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    # 4. Punctuation TIDAK DIHAPUS SEMUANYA, hanya merapikan karakter aneh saja
    # Kita pertahankan huruf, angka, spasi, serta tanda baca krusial hukum: / ( ) , . -
    cleaned_text = re.sub(r'[^a-z0-9\s\/\(\)\,\.\-]', '', cleaned_text)
    
    len_cleaned = len(cleaned_text)
    ratio = (len_cleaned / len_original) * 100 if len_original > 0 else 0
    status = "VALID (>= 80% isi tersedia)" if ratio >= 80 else "WARNING (terlalu banyak terhapus)"
    
    # Simpan kembali teks yang sudah bersih (menimpa file lama dengan aman)
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(cleaned_text)
    
    # Cetak log singkat
    log_info = f"File: {txt_file}\n - Karakter Asli: {len_original}\n - Karakter Bersih: {len_cleaned}\n - Rasio Keutuhan: {ratio:.2f}%\n - Status: {status}\n" + "-"*40
    print(f"✅ Selesai memproses: {txt_file} ({ratio:.2f}% data dipertahankan)")
    log_lines.append(log_info)

# Simpan log ke file
# --- PERBAIKAN PATH & MODE APPEND ---
# 1. Arahkan ke folder logs yang sesuai dengan struktur foldermu
path_log_utama = "../logs/cleaning.log"

# 2. Gunakan mode "a" (Append) agar log baru menyambung di bawah log lama
with open(path_log_utama, "a", encoding="utf-8") as log_file:
    # Tambahkan penanda rerun agar log lebih rapi dan mudah dibaca
    log_file.write("\n\n=== RERUN PREPROCESSING: ADAPTIF PASAL & PIHAK ===\n")
    log_file.write("\n".join(log_lines))

print(f"\nProses Preprocessing Ulang Selesai! Log berhasil digabung ke: {path_log_utama}")

print("\nProses Preprocessing Ulang Selesai!")

Menemukan 36 file teks untuk dibersihkan.

✅ Selesai memproses: case_001.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_002.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_003.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_004.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_005.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_006.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_007.txt (99.98% data dipertahankan)
✅ Selesai memproses: case_008.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_009.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_010.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_011.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_012.txt (99.93% data dipertahankan)
✅ Selesai memproses: case_013.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_014.txt (100.00% data dipertahankan)
✅ Selesai memproses: case_015.txt (100.00% data dipertahankan)
✅ Selesai memp